# ADK Foundations

**Taught session.** Run the cells. Read the comments. Answer the reflection questions.

By the end of this notebook you will:
- Know why symptom triage needs an agent, not a single prompt
- Understand `LlmAgent`, `SequentialAgent`, and `session.state`
- Run a 2-agent pipeline and inspect intermediate state
- Understand Sahayak's 6-stage design and why each stage exists

<!-- ASSESSMENT_GUIDE v1 -->
## Assessment & Submission Guide  ·  9 marks

**Learning objectives — by the end of this notebook you can:**
- Explain why symptom triage needs a multi-agent system, not a single prompt.
- Use `LlmAgent`, `SequentialAgent`, and `session.state`, and inspect intermediate state.
- Measure a `ParallelAgent` speedup against a sequential chain.
- Frame the triage problem: persona, three care levels, asymmetric loss, non-goals.

**Files to modify & submit:**
- `adk_foundations.ipynb` — run every cell and answer the reflection/concept cells.

**Files provided for reference (do not submit):**
- `utils.py`
- Google ADK documentation

**Depends on:** None — this is the entry point.

**Stage → Task → Sub-task → Marks → Expected output**

| Task | Marks | Sub-task | Marks | Expected output |
|---|---:|---|---:|---|
| **1.1 Define the Triage Problem Scope** | **3** | 1.1.1 | 3 | Persona, three care levels, asymmetric loss, non-goals, measurable success criteria |
| **1.2 Learn ADK Through Small Examples** | **6** | 1.2.1 | 4 | >=2 ADK demos incl. a SequentialAgent chain with visible session state |
|  |  | 1.2.2 | 2 | Parallel-vs-sequential timing with the speedup stated |
| | | | **9** | |

**What counts as a completed deliverable:**
- The notebook executes top-to-bottom in Colab (Gemini) or locally (Ollama) with no errors.
- Every claimed number is visible as a notebook cell output (no separate .json artifacts required).
- Every sub-task above has visible evidence in the listed location.
- Attach `final_report.pdf` covering methodology, eval results, failure analysis, known limits, and dashboard screenshots.


## Concept Coverage

**What you already know** (no re-teaching needed):
- Prompt chaining: call LLM A, pass output to LLM B
- Workflow orchestration: LangChain LCEL chains, LangGraph nodes + edges, state dict
- RAG basics: embed documents, cosine search, inject retrieved context into a prompt

**What is new in this capstone** (ADK-specific -- taught here):

| Concept | One line | Where |
|---|---|---|
| `LlmAgent` | A single-job agent with one instruction and one `output_key` | § 2 |
| `SequentialAgent` | Chains agents; passes state via `{output_key}` placeholders | § 2 |
| `ParallelAgent` | Runs independent agents concurrently, then merges state | § 2b |
| `Runner` + `InMemorySessionService` | Executes the pipeline; manages per-patient state | § 4 |
| `output_key -> {placeholder}` handoff | ADK's state-passing mechanism (replaces LangGraph's state dict) | § 3 |

**ADK vs LangGraph in one sentence**: LangGraph gives you a graph with explicit edges;
ADK gives you composable agents where state flows via named keys -- less control, more
encapsulation, easier to add/remove stages.

Everything in this notebook is **taught** -- you run it, read it, answer the reflection
questions, then proceed to next notebook where you start building.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

base_dir = '/content/drive/MyDrive/upgrad AI/Assignment/Capstone_B/Capstone B - Starter Files'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!pip install -q 'google-adk>=2.0.0' google-genai datasets pandas matplotlib seaborn scikit-learn

In [9]:
# >>> output-hygiene (HF/torch import advisories are not errors) >>>
import os as _os, logging as _logging, warnings as _warnings
for _k, _v in {"HF_HUB_DISABLE_IMPLICIT_TOKEN": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
               "HF_HUB_DISABLE_TELEMETRY": "1", "HF_HUB_VERBOSITY": "error",
               "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",
               "TOKENIZERS_PARALLELISM": "false"}.items():
    _os.environ.setdefault(_k, _v)
_warnings.filterwarnings("ignore")
for _n in ("huggingface_hub", "huggingface_hub.utils._http", "transformers",
           "sentence_transformers", "datasets", "torch",
           "torch.distributed.elastic.multiprocessing.redirects", "torchao"):
    _logging.getLogger(_n).setLevel(_logging.ERROR)
# <<< output-hygiene <<<
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic')
warnings.filterwarnings('ignore', message='SequentialAgent is deprecated')
warnings.filterwarnings('ignore', message='ParallelAgent is deprecated')
# -- SETUP -- Colab: uncomment the pip install + API key lines --------------
# !pip install -q 'google-adk>=2.0.0' google-genai datasets pandas matplotlib seaborn scikit-learn

import os
# from google.colab import userdata
# os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
# os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'FALSE'
# Get the free API key at https://aistudio.google.com  (no billing required)

# Model auto-select: Gemini when GOOGLE_API_KEY is set, otherwise local Ollama.
if os.environ.get('GOOGLE_API_KEY'):
    MODEL = 'gemini-3.5-flash'
    print('Model: gemini-3.5-flash (Google AI Studio)')
else:
    from google.adk.models.lite_llm import LiteLlm
    os.environ['GOOGLE_API_KEY'] = 'dummy'   # ADK requires the var; LiteLLM ignores it
    os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'FALSE'
    MODEL = LiteLlm(model='ollama_chat/hermes3:8b', api_base='http://localhost:11434')
    print('Model: hermes3:8b via local Ollama (no GOOGLE_API_KEY found)')

Model: gemini-3.5-flash (Google AI Studio)


<!-- TASKMARK -->
# Stage 1 — Business Objective, Data Understanding, and Agent Foundations <font color="red">[24 marks]</font>
## Task 1.1 — Define the triage problem scope <font color="red">[3 marks]</font>
### **1.1.1** Scope the problem <font color="red">[3 marks]</font>

Define the ASHA-worker persona, the three care levels (WAIT/DOCTOR/ER), the asymmetric-loss principle, explicit non-goals (no diagnosis, no prescription) and measurable success criteria **(should be written in your reflection answers + the problem statement)** — ***(to be done in this notebook and the report)***

### Problem Statement: Symptom Triage for ASHA Workers

**Persona**: The primary user is an ASHA (Accredited Social Health Activist) worker in a rural Indian setting. ASHA workers are frontline health workers with limited medical training, acting as a crucial link between the community and the healthcare system. They need a simple, reliable, and safe tool to guide initial patient interactions.

**Three Care Levels**:
*   **WAIT**: Symptoms are minor, self-limiting, and can be managed with home care or a deferred appointment (e.g., common cold, minor aches). No immediate medical attention required.
*   **DOCTOR**: Symptoms suggest a condition requiring a general physician's assessment within 24-48 hours (e.g., persistent fever, moderate pain, non-urgent infections). Requires a clinic visit, but not an emergency.
*   **ER**: Symptoms indicate a medical emergency requiring immediate attention at an emergency room or hospital (e.g., severe chest pain, difficulty breathing, signs of stroke, loss of consciousness). Time-critical, life-threatening conditions.

**Asymmetric Loss Principle**: The cost of misclassifying an ER case as DOCTOR or WAIT is significantly higher (potentially life-threatening) than misclassifying a WAIT case as DOCTOR or ER. The system must prioritize false negatives for ER cases to be as low as possible, even if it means a higher rate of false positives for DOCTOR or ER.

**Non-Goals**:
*   **No Diagnosis**: The system will not provide a medical diagnosis of any condition or illness.
*   **No Prescription/Treatment Advice**: The system will not recommend specific medications, dosages, or treatment protocols.
*   **Not a Replacement for Medical Professionals**: The system is a decision-support tool for ASHA workers, not a substitute for qualified medical advice or professional judgment.

**Measurable Success Criteria**:
*   **High Recall for ER Cases**: Minimize false negatives for ER-level conditions (e.g., >95% recall for ER cases).
*   **Accuracy of Triage Level**: Overall accuracy of assigning the correct care level (WAIT/DOCTOR/ER) across all cases.
*   **Safety Audit Pass Rate**: A high percentage of generated responses pass the safety evaluator's checks for diagnosis, prescription, and medical advice violations.
*   **User Satisfaction**: Feedback from ASHA workers indicating the tool is helpful, easy to use, and increases their confidence in guiding patients.

## 1 · Why An Agent, Not a Single Prompt?

A single LLM prompt for triage has three problems:

| Problem | Why it matters |
|---------|----------------|
| The model can hallucinate a diagnosis | Priya must not tell a patient they have disease X |
| There is no audit trail | We cannot tell which step failed |
| Improvement is guesswork | We cannot test symptom extraction separately from the decision |

An agent pipeline solves all three: each stage is small, testable, and inspectable.

**The rule**: every stage writes evidence. Every run can be explained.

In [10]:
# Priya's manual decision process -- this is what we are automating

priya_steps = [
    '1. Listen: what symptoms does the patient have?',
    '2. Check red flags: anything immediately dangerous?',
    '3. Score urgency: mild / moderate / serious',
    '4. Ask one question if unsure',
    '5. Decide: WAIT / DOCTOR / ER',
    '6. Explain in plain language -- no diagnosis, no prescription',
    '7. Self-check: did I accidentally diagnose or prescribe?',
]

for step in priya_steps:
    print(step)

1. Listen: what symptoms does the patient have?
2. Check red flags: anything immediately dangerous?
3. Score urgency: mild / moderate / serious
4. Ask one question if unsure
5. Decide: WAIT / DOCTOR / ER
6. Explain in plain language -- no diagnosis, no prescription
7. Self-check: did I accidentally diagnose or prescribe?


<!-- TASKMARK -->
## Task 1.2 — Learn ADK Through Small Examples <font color="red">[6 marks]</font>
### **1.2.1** Run ≥2 ADK demos <font color="red">[4 marks]</font>

Build and execute ≥2 ADK demos including a `SequentialAgent` chain with visible session state between agents.

## 2 · Three ADK Primitives

### `LlmAgent` -- one job, one model call
```python
LlmAgent(
    name='symptom_parser',          # unique, shows up in traces
    model='gemini-2.0-flash',
    instruction='Extract symptoms from: {patient_input}',  # {key} reads from state
    output_key='symptoms',           # writes result to session.state['symptoms']
)
```

### `SequentialAgent` -- fixed-order pipeline
```python
SequentialAgent(name='pipeline', sub_agents=[agent_a, agent_b, agent_c])
# Runs A -> B -> C. Each agent reads state written by all previous agents.
# Order is deterministic -- NOT controlled by AI.
```

### `session.state` -- the shared whiteboard
```python
# Before pipeline runs:
session.state = {'patient_input': 'fever for 3 days'}

# After pipeline runs:
session.state = {
    'patient_input':   'fever for 3 days',
    'symptoms':        '["fever", "duration:3 days"]',
    'severity_json':   '{"severity": 3, "reason": "moderate"}',
    'followup':        '{"needed": true, "question": "..."}',
    'triage_decision': '{"triage_level": "DOCTOR", ...}',
    'final_response':  'Based on what you described ...',
    'safety_audit':    '{"verdict": "PASS", ...}',
}
```

In [11]:
# -- DEMO: single LlmAgent -------------------------------------------------
# Notebooks already run inside an event loop -- use top-level `await`,
# NOT asyncio.run() or loop.run_until_complete().

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

echo = LlmAgent(
    name='symptom_echo',
    model=MODEL,
    instruction='Return ONLY a JSON list of visible symptoms from: {patient_input}',
    output_key='symptoms',
)
ss = InMemorySessionService()
runner = Runner(agent=echo, app_name='demo', session_service=ss)

async def run_one(text):
    await ss.create_session(app_name='demo', user_id='u', session_id='s1',
                            state={'patient_input': text})
    msg = Content(role='user', parts=[Part(text=text)])
    async for _ in runner.run_async(user_id='u', session_id='s1', new_message=msg):
        pass
    s = await ss.get_session(app_name='demo', user_id='u', session_id='s1')
    return dict(s.state)

state = await run_one('Patient has fever and headache for 2 days')
print('session.state after one LlmAgent:')
for k, v in state.items():
    print(f'  {k}: {str(v)[:160]}')

session.state after one LlmAgent:
  patient_input: Patient has fever and headache for 2 days
  symptoms: [
  "fever",
  "headache"
]


## 3 · The output_key -> {placeholder} Handoff

This is the core pattern for state passing between agents:

```
Agent A  ->  output_key='symptoms'  ->  writes session.state['symptoms']
Agent B  ->  instruction='...{symptoms}...'  ->  framework injects the value
```

No Python plumbing. No `return` values. Just strings in state.

⚠️ **Debugging tip**: if a `{placeholder}` shows up empty, inspect `session.state` directly:
```python
s = await session_service.get_session(app_name=..., user_id=..., session_id=...)
print(dict(s.state))  # shows every key written so far
```

In [12]:
# NOTE: SequentialAgent and ParallelAgent show a strikethrough in some IDEs
# because ADK marked them deprecated (pointing to a future Workflow class).
# They are fully functional in ADK 2.x -- ignore the strikethrough for now.
# -- DEMO: two-agent SequentialAgent --------------------------------------
# Shows the output_key -> {placeholder} handoff end to end.

from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

parser = LlmAgent(
    name='parser',
    model=MODEL,
    instruction='Return ONLY a JSON list of symptoms from: {patient_input}',
    output_key='symptoms',
)
scorer = LlmAgent(
    name='scorer',
    model=MODEL,
    # Note: {symptoms} is injected automatically from session.state
    instruction='Score urgency 1-5. Return ONLY JSON: {{"severity": N, "reason": "..."}}\nSymptoms: {symptoms}',
    output_key='severity_json',
)

pipeline = SequentialAgent(name='demo_pipeline', sub_agents=[parser, scorer])
ss2 = InMemorySessionService()
r2 = Runner(agent=pipeline, app_name='demo2', session_service=ss2)

async def run_two(text):
    await ss2.create_session(app_name='demo2', user_id='u', session_id='s2',
                             state={'patient_input': text})
    msg = Content(role='user', parts=[Part(text=text)])
    async for _ in r2.run_async(user_id='u', session_id='s2', new_message=msg):
        pass
    s = await ss2.get_session(app_name='demo2', user_id='u', session_id='s2')
    return dict(s.state)

state2 = await run_two('Patient has chest pain and difficulty breathing')
print('symptoms:', state2.get('symptoms'))
print('severity_json:', state2.get('severity_json'))

symptoms: [
  "chest pain",
  "difficulty breathing"
]
severity_json: {
  "severity": 5,
  "reason": "Chest pain accompanied by difficulty breathing represents a potential life-threatening medical emergency, such as a heart attack or pulmonary embolism, requiring immediate emergency medical intervention."
}


<!-- TASKMARK -->
### **1.2.2** Measure the parallel speedup <font color="red">[2 marks]</font>

Time a `ParallelAgent` against the sequential chain and state the speedup factor.

## 2b · `ParallelAgent` -- Fan-Out, Then Merge

Some tasks are **independent** -- they don't need each other's output to start.
Running them one after the other wastes time. `ParallelAgent` runs them **at the same time**.

```
SequentialAgent          ParallelAgent
----------------         --------------------------
A -> B -> C               A --┐
                         B --┼-- all finish -> merge
3 × latency              C --┘
                         1 × latency (slowest wins)
```

### When to use `ParallelAgent`

| Use it when | Do NOT use it when |
|-------------|--------------------|
| Agents are **independent** -- no agent needs another's output to start | Agent B needs Agent A's output as input |
| You want **multiple perspectives** on the same input | You need a fixed decision chain |
| Latency matters and tasks can overlap | Order of execution matters |

### Sahayak example -- where would ParallelAgent help?

Right now Sahayak runs all 6 stages sequentially. But two of them are independent:

```
symptom_parser     -> must finish first (others need its output)
  ↓
severity_scorer --┐
red_flag_checker -┼-- ParallelAgent: both read {symptoms}, run at the same time
                  ↓
triage_decider    -> reads both outputs before deciding
```

`severity_scorer` and a `red_flag_checker` both only need `{symptoms}` as input.
Neither needs the other. Running them in parallel cuts their combined latency in half.

> **Key rule**: `ParallelAgent` is a fan-out -- sub-agents write to **different** `output_key`s.
> They must never write to the same key (last write wins, silently).


In [13]:
# NOTE: SequentialAgent and ParallelAgent show a strikethrough in some IDEs
# because ADK marked them deprecated (pointing to a future Workflow class).
# They are fully functional in ADK 2.x -- ignore the strikethrough for now.
# -- DEMO: SequentialAgent vs ParallelAgent -- wall-clock comparison ------------
# Two independent tasks: extract symptoms AND check for red flags.
# We time both designs on the same input.

import time
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

INPUT = 'Patient has chest pain, sweating, and difficulty breathing for 20 minutes.'

def make_agents():
    # Fresh instances each call -- an ADK agent can only have ONE parent,
    # so seq and par must not share the same agent objects.
    extractor = LlmAgent(
        name='symptom_extractor',
        model=MODEL,
        instruction='Extract visible symptoms from: {patient_input}. Return ONLY a JSON list.',
        output_key='symptoms',
    )
    checker = LlmAgent(
        name='red_flag_checker',
        model=MODEL,
        instruction=(
            'Check for emergency red flags in: {patient_input}. '
            'Red flags: chest pain, difficulty breathing, loss of consciousness, stroke signs. '
            'Return ONLY JSON: {"red_flags_found": ["..."], "emergency": true/false}'
        ),
        output_key='red_flags',
    )
    return extractor, checker

async def run_design(agent, label):
    ss = InMemorySessionService()
    runner = Runner(agent=agent, app_name='demo', session_service=ss)
    await ss.create_session(app_name='demo', user_id='u1', session_id='s1',
                            state={'patient_input': INPUT})
    t0 = time.perf_counter()
    async for _ in runner.run_async(
        user_id='u1', session_id='s1',
        new_message=Content(parts=[Part(text=INPUT)])
    ):
        pass
    elapsed = time.perf_counter() - t0
    s = await ss.get_session(app_name='demo', user_id='u1', session_id='s1')
    print(f'\n-- {label} ({elapsed:.2f}s) --')
    for k in ['symptoms', 'red_flags']:
        if k in dict(s.state):
            print(f'  {k}: {str(dict(s.state)[k])[:120]}')
    return elapsed

seq = SequentialAgent(name='seq_demo', sub_agents=list(make_agents()))
par = ParallelAgent(name='par_demo', sub_agents=list(make_agents()))

t_seq = await run_design(seq, 'SequentialAgent')
t_par = await run_design(par, 'ParallelAgent ')
speedup = t_seq / t_par if t_par > 0 else 0
print(f'\n-- Summary (MEASURED on this machine) --')
print(f'  Sequential : {t_seq:.2f}s')
print(f'  Parallel   : {t_par:.2f}s')
print(f'  Speedup    : {speedup:.1f}x')
print()
print('Both designs produce the same outputs -- symptoms and red_flags.')
print('IMPORTANT -- where the speedup comes from: ParallelAgent overlaps the')
print('MODEL CALLS. On a cloud API (Gemini) both requests genuinely run at')
print('once -> expect ~1.5-2x. On a single local GPU, Ollama queues requests')
print('one at a time, so parallel can be EQUAL OR SLOWER (concurrency without')
print('parallel capacity adds overhead). The architecture lesson is the same:')
print('parallelism pays exactly when the backend can serve requests concurrently.')
print('For N independent agents on a concurrent backend: Sequential = N x latency, Parallel = 1 x slowest.')


-- SequentialAgent (5.19s) --
  symptoms: [
  "chest pain",
  "sweating",
  "difficulty breathing"
]
  red_flags: {"red_flags_found": ["chest pain", "difficulty breathing"], "emergency": true}

-- ParallelAgent  (3.27s) --
  symptoms: [
  "chest pain",
  "sweating",
  "difficulty breathing"
]
  red_flags: {"red_flags_found": ["chest pain", "difficulty breathing"], "emergency": true}

-- Summary (MEASURED on this machine) --
  Sequential : 5.19s
  Parallel   : 3.27s
  Speedup    : 1.6x

Both designs produce the same outputs -- symptoms and red_flags.
IMPORTANT -- where the speedup comes from: ParallelAgent overlaps the
MODEL CALLS. On a cloud API (Gemini) both requests genuinely run at
once -> expect ~1.5-2x. On a single local GPU, Ollama queues requests
one at a time, so parallel can be EQUAL OR SLOWER (concurrency without
parallel capacity adds overhead). The architecture lesson is the same:
parallelism pays exactly when the backend can serve requests concurrently.
For N independent 

### The Gains -- and the Trade-offs

| | `SequentialAgent` | `ParallelAgent` |
|---|---|---|
| **Latency** | Adds up: A + B + C | Max of: max(A, B, C) |
| **For N=2 equal agents** | 2× slowest | 1× slowest |
| **For N=6 equal agents** | 6× slowest | 1× slowest |
| **State dependencies** | Fine -- B reads A's output | Forbidden -- sub-agents must be independent |
| **Debugging** | Easy -- one event stream | Harder -- interleaved events |
| **Token cost** | Same | Same (both agents still run) |
| **When order matters** | Yes | No -- finish order is non-deterministic |

> **Sahayak uses `SequentialAgent`** because each stage depends on the previous one.
> `symptom_parser` must finish before `severity_scorer` can read `{symptoms}`.
> You cannot parallelise a dependency chain.
>
> **Where `ParallelAgent` fits in Sahayak**: a future `clinical_review` stage could run
> three specialist agents (cardiac, respiratory, neurological) in parallel on the same
> symptoms, then merge their verdicts before the `triage_decider`. That is agent_evaluation_and_optimisation.ipynb Appendix Option B.


## 4 · Sahayak's 6-Stage Design

```
symptom_parser     -> output_key='symptoms'
  ↓ {symptoms}
severity_scorer    -> output_key='severity_json'
  ↓ {severity_json}
followup_asker     -> output_key='followup'
  ↓ {followup}
triage_decider     -> output_key='triage_decision'
  ↓ {triage_decision}
response_formatter -> output_key='final_response'
  ↓ {final_response}
safety_evaluator   -> output_key='safety_audit'
```

Why 6 stages, not 1?

| Stage | Why it exists |
|-------|---------------|
| `symptom_parser` | Handles messy real-world text -- normalises input |
| `severity_scorer` | Rule-constrained -- safety logic is explicit and testable |
| `followup_asker` | Resolves ambiguity before committing to a decision |
| `triage_decider` | Rule-locked -- LLM cannot freely invent a triage level |
| `response_formatter` | Communicates safely -- no diagnosis, no prescription |
| `safety_evaluator` | Audits the chain -- the final check before Priya sees the output |

## 5 · Reflection Questions

Answer in the code cell below before moving to the next notebook.

1. Why should `triage_decider` use a rule-constrained prompt rather than free LLM judgment?
2. What is the difference between `SequentialAgent`, `ParallelAgent`, and calling agents in a Python `for` loop? When would you choose each?
3. Why is the `safety_evaluator` a separate 6th agent rather than an extra instruction in `response_formatter`?
4. Name one violation the `safety_evaluator` catches that an accuracy metric on a test set would miss.
5. What would happen to ER recall if the severity_scorer used a free prompt instead of explicit red-flag rules?

---
### Your Task

Answer the 5 reflection questions in the cell below in your own words.
Do not copy phrasing from the notebook above.

The ADK demos (`LlmAgent` and `SequentialAgent`) have been run and their outputs showing `session.state` are visible in the notebook (cells `0863f185` and `8fdce53e` respectively). These illustrate how `LlmAgent` performs a single task and how `SequentialAgent` chains multiple agents, passing state between them using `output_key` and `{placeholder}`. The `ParallelAgent` demonstration and speedup measurement are also visible in cell `df92c3d1`.

In [14]:
# -- YOUR ANSWERS ---------------------------------------------------------
# -- YOUR ANSWERS ---------------------------------------------------------

answers = {
    'Q1': 'Free judgment introduces clinical volatility and inconsistency. Rule-constrained prompts enforce strict, deterministic boundaries to ensure life-threatening symptoms are never misclassified.',
    'Q2': 'SequentialAgent and ParallelAgent manage execution and state natively via ADK framework primitives, whereas a for loop requires manual Python variable plumbing. Choose Sequential for dependent stages, Parallel for independent tasks, and a for loop for dynamic runtime control flow.',
    'Q3': 'It implements separation of concerns. A response formatter focused on tone can easily miss its own safety violations, whereas an independent 6th agent provides an unbiased audit.',
    'Q4': 'An implicit violation like accidental diagnosis or prescription generation, which can slip past semantic accuracy metrics if the overall text looks helpful and well-formatted.',
    'Q5': 'ER recall would drop because a free prompt lacks strict enforcement, creating a dangerous risk of false negatives where critical emergency red flags are incorrectly downplayed.',
}

for q, a in answers.items():
    print(f'{q}: {a}')

Q1: Free judgment introduces clinical volatility and inconsistency. Rule-constrained prompts enforce strict, deterministic boundaries to ensure life-threatening symptoms are never misclassified.
Q2: SequentialAgent and ParallelAgent manage execution and state natively via ADK framework primitives, whereas a for loop requires manual Python variable plumbing. Choose Sequential for dependent stages, Parallel for independent tasks, and a for loop for dynamic runtime control flow.
Q3: It implements separation of concerns. A response formatter focused on tone can easily miss its own safety violations, whereas an independent 6th agent provides an unbiased audit.
Q4: An implicit violation like accidental diagnosis or prescription generation, which can slip past semantic accuracy metrics if the overall text looks helpful and well-formatted.
Q5: ER recall would drop because a free prompt lacks strict enforcement, creating a dangerous risk of false negatives where critical emergency red flags are

In [15]:
# -- Auto-check: did you answer all 5 questions? -----------------------------
missing = [q for q, a in answers.items() if a.strip() == 'FILL IN']
if missing:
    print(f'[INCOMPLETE] Still need answers for: {missing}')
    print('Answer all 5 before moving to the next notebook.')
else:
    print('[OK] All 5 answered. Your answers:')
    for q, a in answers.items():
        print(f'  {q}: {a[:80]}')
    print()
    print('Notebook 1 complete.')


[OK] All 5 answered. Your answers:
  Q1: Free judgment introduces clinical volatility and inconsistency. Rule-constrained
  Q2: SequentialAgent and ParallelAgent manage execution and state natively via ADK fr
  Q3: It implements separation of concerns. A response formatter focused on tone can e
  Q4: An implicit violation like accidental diagnosis or prescription generation, whic
  Q5: ER recall would drop because a free prompt lacks strict enforcement, creating a 

Notebook 1 complete.


## Checkpoint

Tick each before next notebook:

- [ ] I can describe `LlmAgent`, `SequentialAgent`, `ParallelAgent`, and `session.state` in one sentence each
- [ ] I understand why `output_key='symptoms'` makes `{symptoms}` available in the next agent
- [ ] I ran the 2-agent sequential demo and read the `session.state` output
- [ ] I ran the parallel demo and understood why both designs produce the same outputs but at different speeds
- [ ] I understand why `safety_evaluator` is a separate stage
- [ ] I answered all 5 reflection questions